# Notebook: BCI_42_CarDet_Criterios_Salida
*********************************************************************************

## Informacion del Notebook

### Encabezado
**************************************************************************
* Nombre: BCI_42_CarDet_Criterios_Salida.ipynb
* Ruta: https://adb-5512273708018582.2.azuredatabricks.net/?o=5512273708018582#notebook/2997520011901275
* Autor: Gabriel MartÍnez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 12/08/2022
* Descripcion: Evaluacion criterios de salida del periodo actual
* Documentacion:
***************************************************************************

### Mantenciones
**************************************************************************
#### Mantención Nro: 1
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 10/02/2025 
* Descripción: Se agrega nuevos Salidas (44 - LIR; 26 - InterSegmento)     
***************************************************************************

**************************************************************************
#### Mantención Nro: 2
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 10/04/2025 
* Descripción: Se elimina Salida LIR (44 - LIR)     
***************************************************************************

### Tablas Entrada y Salida
**************************************************************************
#### Tablas Entrada: 
* {base_silver_x}.tbl_cd_cartdet_crit_sal_ope_eval
***************************************************************************
#### Tablas Salida: 
* {base_silver_x}.tbl_cd_cartdet_crit_sal_crit
***************************************************************************


## Carga Dependencias

### Carga funciones comunes

In [0]:
%run "./Funciones_Comunes"

# Notebook: Funciones_Comunes
**************************************************************************

## Informacion del Notebook 

### Encabezado
**************************************************************************
* Nombre: Funciones_Comunes.ipynb
* Ruta: https://adb-5512273708018582.2.azuredatabricks.net/?o=5512273708018582#notebook/3570959530595695
* Autor: Gabriel Martínez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 23/09/2023
* Descripcion: Notebook con funciones genéricas que pueden ser usadas por otros notebooks.
* Documentacion:
***************************************************************************

### Mantenciones
**************************************************************************
#### Mantención Nro: 1
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 15/02/2025 
* Descripción: Se cambio el metodo de cancelacion utilizando el comando (raise) y se incorporada la funcion de ir a buscar el ultimo dia calendario. Tambien se agrego una nueva funcion (obtener_estados_tablas).  
***************************************************************************
#### Mantención Nro: 2
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Jonatan Cancino
* Fecha: 25/04/2025 
* Descripción: Se modifico la funcion extension_archivos para que cuando la vigencia sea previa, asigne extencion .PRV.  
***************************************************************************
#### Mantención Nro: 3
* Autor: Gabriel Martinez (SimpleData) - Ing. SW BCI: Claudia Yañez
* Fecha: 08/07/2025 
* Descripción: Se realiza una mejora en la funcion mostrar_variacion_criterio.  
***************************************************************************

## Carga librerias

...

### obtener archivo periodo anterior

### concatena archivos

###primer_dia_mes_sig

###Calcula fecha X meses atras

## FIN definición de funciones

## Parámetros

### Setea Parámetros

In [0]:

dbutils.widgets.text("fecha_w","","01-Fecha:")
dbutils.widgets.text("bd_silver_w","","03-Nombre BD Silver:")

fecha_x = dbutils.widgets.get("fecha_w")
base_silver_x = dbutils.widgets.get("bd_silver_w")

spark.conf.set("bci.fecha", fecha_x)
spark.conf.set("bci.dbnamesilver", base_silver_x)

print(f"Fecha de Proceso actual: [fecha_x] {fecha_x}")
print(f"Nombre BD Silver: [base_silver_x] {base_silver_x}")


Fecha de Proceso actual: [fecha_x] 20250930
Nombre BD Silver: [base_silver_x] dsr_gld_bciwork_db


### Valida parámetros

In [0]:
valida_parametro(fecha_x)

In [0]:
valida_parametro(base_silver_x)

## INICIO Proceso extraccion y transformacion
--------------------------------------
- Por cada fuente que se utilice se debe:
     - Titulo: generar un titulo generico, con nombre fuente, y descripcion del proposito de la extraccion
     - Extraer: para el periodo, o rango de fecha que se necesita la iformacion. Debe tener el prefijo tmp_EXT_{nombrefuente}
     - Transformar: generar la informacion necesaria para la salida final. Se pueden generar mas de una tabla temporal para llegar al resultado final. Debe tener el prefijo tmp_RES_{nombre}_correlativo


### Parametrizacion
---
* define y asigna valores a los parametros


In [0]:
#Parametria interna notebook
p_critdet = 2,27,62,40,41,67,60,61,63,64,65,66,30,26
print(f"p_critdet: {p_critdet}")

p_critdet: (2, 27, 62, 40, 41, 67, 60, 61, 63, 64, 65, 66, 30, 26)


### Extrae Evaluaciones  
--------------------------------------
- Se extraen todas las evaluaciones positivas 

In [0]:
paso_query10 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_EXT_tbl_cartdet_crit_sal_ope_eval AS
SELECT 
        b.periodo_cierre,
        b.fecha_cierre,
        b.tipo_proceso,
        b.segmento,
        b.operacion,
        b.tipo_operacion,
        b.sistema,
        b.rut_cliente,
        b.dv_rut_cliente,
        b.nombre_campo,
        b.valor_campo,
        b.condicion_regla,
        b.valor_regla,
        b.flag_resultado_regla,
        b.cod_evaluacion
FROM
  {base_silver_x}.tbl_cd_cartdet_crit_sal_ope_eval b
WHERE 
    b.fecha_cierre =   {fecha_x} 
AND b.flag_resultado_regla = 1
QUALIFY  ROW_NUMBER() OVER(PARTITION BY b.operacion, b.sistema, b.cod_evaluacion ORDER BY b.fecha_cierre DESC) =1
"""


In [0]:
sql_safe(paso_query10)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_EXT_tbl_cartdet_crit_sal_ope_eval AS
SELECT 
        b.periodo_cierre,
        b.fecha_cierre,
        b.tipo_proceso,
        b.segmento,
        b.operacion,
        b.tipo_operacion,
        b.sistema,
        b.rut_cliente,
        b.dv_rut_cliente,
        b.nombre_campo,
        b.valor_campo,
        b.condicion_regla,
        b.valor_regla,
        b.flag_resultado_regla,
        b.cod_evaluacion
FROM
  dsr_gld_bciwork_db.tbl_cd_cartdet_crit_sal_ope_eval b
WHERE 
    b.fecha_cierre =   20250930 
AND b.flag_resultado_regla = 1
QUALIFY  ROW_NUMBER() OVER(PARTITION BY b.operacion, b.sistema, b.cod_evaluacion ORDER BY b.fecha_cierre DESC) =1



DataFrame[]

### Matriz con evaluciones por operacion
--------------------------------------
- Genera matriz por operacion con indicador de evaluacion indicando si cumple o no cumple dicha evaluacion

In [0]:
paso_query20 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_matriz_cartdet_crit_sal_ope_eval AS
SELECT
     periodo_cierre
    ,fecha_cierre
    ,tipo_proceso
    ,operacion
    ,sistema
    ,rut_cliente
    ,dv_rut_cliente
    ,tipo_operacion
    ,segmento
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S01' THEN 1 ELSE 0 END) AS IND_S01
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S02' THEN 1 ELSE 0 END) AS IND_S02
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S03' THEN 1 ELSE 0 END) AS IND_S03
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S04' THEN 1 ELSE 0 END) AS IND_S04
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S05' THEN 1 ELSE 0 END) AS IND_S05
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S06' THEN 1 ELSE 0 END) AS IND_S06
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S07' THEN 1 ELSE 0 END) AS IND_S07
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S08' THEN 1 ELSE 0 END) AS IND_S08
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S09' THEN 1 ELSE 0 END) AS IND_S09
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S10' THEN 1 ELSE 0 END) AS IND_S10
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S11' THEN 1 ELSE 0 END) AS IND_S11
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S12' THEN 1 ELSE 0 END) AS IND_S12
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S13' THEN 1 ELSE 0 END) AS IND_S13	
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S14' THEN 1 ELSE 0 END) AS IND_S14
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S15' THEN 1 ELSE 0 END) AS IND_S15    
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'N01' THEN 1 ELSE 0 END) AS IND_N01
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'N02' THEN 1 ELSE 0 END) AS IND_N02
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'N03' THEN 1 ELSE 0 END) AS IND_N03
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'N04' THEN 1 ELSE 0 END) AS IND_N04
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'N05' THEN 1 ELSE 0 END) AS IND_N05
FROM
    tmp_EXT_tbl_cartdet_crit_sal_ope_eval A
GROUP BY 
   1,2,3,4,5,6,7,8,9
"""   

In [0]:
sql_safe(paso_query20)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_matriz_cartdet_crit_sal_ope_eval AS
SELECT
     periodo_cierre
    ,fecha_cierre
    ,tipo_proceso
    ,operacion
    ,sistema
    ,rut_cliente
    ,dv_rut_cliente
    ,tipo_operacion
    ,segmento
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S01' THEN 1 ELSE 0 END) AS IND_S01
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S02' THEN 1 ELSE 0 END) AS IND_S02
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S03' THEN 1 ELSE 0 END) AS IND_S03
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S04' THEN 1 ELSE 0 END) AS IND_S04
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S05' THEN 1 ELSE 0 END) AS IND_S05
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S06' THEN 1 ELSE 0 END) AS IND_S06
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S07' THEN 1 ELSE 0 END) AS IND_S07
    ,MAX(CASE WHEN IFNULL(A.cod_evaluacion,'XXX') = 'S08' THEN 1 ELSE 0 END) AS IND_S08
    ,MAX(CASE WHEN IFNULL(A.cod_evalu

DataFrame[]

### Salida Deterioro operaciones de clientes individuales (2)
--------------------------------------
- Genera registros para operaciones deterioradas individualmente
- Clientes INDIVIDUAL sin calificacion de deterioro (Evaluacion S01 and S02), ENTONCES 1
- Clientes GRUPAL ENTONCES 1
- S01: cliente individual 
- S02: cliente NO tiene una calificacion deteriorada 

In [0]:
paso_query30 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_1 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,2                        AS criterio_salida
,CASE 
  WHEN IFNULL(A.IND_S01,0)=1 AND IFNULL(A.IND_S02,0)=1  THEN 1
  WHEN IFNULL(A.IND_S01,0)=0 THEN 1 
  ELSE 0 
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A
"""


In [0]:
sql_safe(paso_query30)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_1 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,2                        AS criterio_salida
,CASE 
  WHEN IFNULL(A.IND_S01,0)=1 AND IFNULL(A.IND_S02,0)=1  THEN 1
  WHEN IFNULL(A.IND_S01,0)=0 THEN 1 
  ELSE 0 
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A



DataFrame[]

### Salida Deterioro operaciones con saldo ifrs total cero (27)
--------------------------------------
- Operaciones que cumplen saldo total ifrs cero y no es una excepcion
- S03: Operacion con saldo igual o menor a  0.0
- N01: operacion es excepcion


In [0]:
paso_query35 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_2 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento     
,27                     AS criterio_salida
,CASE 
  WHEN IFNULL(A.IND_S03,0) = 1  AND IFNULL(A.IND_N01,0) = 0 AND IFNULL(A.IND_N02,0) = 0 AND IFNULL(A.IND_N03,0) = 0 AND IFNULL(A.IND_N04,0) = 0 AND IFNULL(A.IND_N05,0) = 0
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A
"""


In [0]:
sql_safe(paso_query35)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_2 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento     
,27                     AS criterio_salida
,CASE 
  WHEN IFNULL(A.IND_S03,0) = 1  AND IFNULL(A.IND_N01,0) = 0 AND IFNULL(A.IND_N02,0) = 0 AND IFNULL(A.IND_N03,0) = 0 AND IFNULL(A.IND_N04,0) = 0 AND IFNULL(A.IND_N05,0) = 0
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A



DataFrame[]

### Salida Deterioro operaciones deterioradas mes anterior y que no se informan en mes actual (30)
--------------------------------------
- Operaciones deteriorada mes anterior y no informadas en mes actual
- S14: Operacion deterioradas periodo anterior no existe en periodo actual 


In [0]:
paso_query40 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_3 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,30                       AS criterio_salida
,CASE 
  WHEN IFNULL(A.IND_S14,0) = 1 
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A
"""

In [0]:
sql_safe(paso_query40)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_3 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,30                       AS criterio_salida
,CASE 
  WHEN IFNULL(A.IND_S14,0) = 1 
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A



DataFrame[]

### Salida Deterioro operaciones no informadas por SSFF (40)
--------------------------------------
- Operaciones no deterioradas en SSFF
- S05: NO Exista en tabla de deterioro SSFF 


In [0]:
paso_query45 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_4 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,40                     AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S05,0) = 1   
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A
"""

In [0]:
sql_safe(paso_query45)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_4 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,40                     AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S05,0) = 1   
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A



DataFrame[]

### Salida Deterioro operaciones no informadas por Factoring (41)
--------------------------------------
- Operaciones no deterioradas en Factoring
- S05: NO Exista en tabla de deterioro Factoring 


In [0]:
paso_query50 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_5 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,41                     AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S06,0) = 1   
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A
"""

In [0]:
sql_safe(paso_query50)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_5 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,41                     AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S06,0) = 1   
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A



DataFrame[]

### Salida Deterioro operaciones con mora menor a (60)
--------------------------------------
- Operaciones con mora menor a
- S08: Maximo dias de mora cliente menor o igual a


In [0]:
paso_query55 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_7 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,60                     AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S08,0) = 1   
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A
"""

In [0]:
sql_safe(paso_query55)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_7 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,60                     AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S08,0) = 1   
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A



DataFrame[]

### Salida Deterioro pagos consecutivos (61)
--------------------------------------
- Operaciones con mas de x pagos consecutivos
- S09: Cantidad de pagos consecutivos mayor o igual a


In [0]:
paso_query60 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_8 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,61                     AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S09,0) = 1   
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A
"""

In [0]:
sql_safe(paso_query60)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_8 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,61                     AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S09,0) = 1   
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A



DataFrame[]

### Salida Deterioro sin mora sbif (62)
--------------------------------------
- Operaciones de clientes sin mora sbif
- S04:  Deuda morosa sbif menor o igual a 


In [0]:
paso_query65 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_9 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,62                       AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S04,0) = 1  
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A
"""

In [0]:
sql_safe(paso_query65)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_9 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,62                       AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S04,0) = 1  
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A



DataFrame[]

### Salida Deterioro operacion con pagos parciales (63)
--------------------------------------
- Operaciones con pagos parciales
- S04:  Deuda morosa sbif menor o igual a 


In [0]:
paso_query70 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_10 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,63                   AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S10,0) = 1  
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A
"""

In [0]:
sql_safe(paso_query70)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_10 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,63                   AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S10,0) = 1  
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A



DataFrame[]

### Salida Deterioro operacion sin refinanciamiento (64)
--------------------------------------
- Operaciones sin refinanciamiento
- S11:  No tenga renegociados ni curse bajo mora 


In [0]:
paso_query75 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_11 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,64                     AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S11,0) = 1  
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A
"""

In [0]:
sql_safe(paso_query75)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_11 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,64                     AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S11,0) = 1  
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A



DataFrame[]

### Salida Deterioro operacion con amortizacion de capital (65)
--------------------------------------
- Operaciones con amortizacion de capital o pagos parciales
- S12:  Saldo capital periodo actual sea menor al saldo periodo anterior 


In [0]:
paso_query80 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_12 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,65                     AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S12,0) = 1  
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A
"""

In [0]:
sql_safe(paso_query80)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_12 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,65                     AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S12,0) = 1  
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A



DataFrame[]

### Salida Deterioro minimo de meses en deterioro (66)
--------------------------------------
- Operaciones con un minimo de meses en deterioro
- S13:  Si numero de meses es mayor o igual a 


In [0]:
paso_query85 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_13 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,66                     AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S13,0) = 1  
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A
"""

In [0]:
sql_safe(paso_query85)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_13 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,66                     AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S13,0) = 1  
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A



DataFrame[]

### Salida Deterioro clientes no LIR (67)
--------------------------------------
- Operaciones de los clientes que no son LIR o cuya antiguedad sea mayor a 12 meses
- S07: Cliente no es LIR entonces cumple regla
- S15: Cliente es LIR pero tiene antiguedad lir mayor a 12 meses entonces cumple regla


In [0]:
paso_query90 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_14 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,67                       AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S07,0) = 1 THEN 1  
  WHEN  IFNULL(A.IND_S07,0) = 0 AND IFNULL(A.IND_S15,0) = 1 THEN 1  
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A
"""

In [0]:
sql_safe(paso_query90)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_14 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento      
,67                       AS criterio_salida
,CASE 
  WHEN  IFNULL(A.IND_S07,0) = 1 THEN 1  
  WHEN  IFNULL(A.IND_S07,0) = 0 AND IFNULL(A.IND_S15,0) = 1 THEN 1  
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A



DataFrame[]

### Salida Deterioro por Intersegmento (26)
--------------------------------------
- Cliente con operaciones Individual con condicion de salida y operacion Grupal con deterioro.
- Deber cumplir con los siguientes criterios de salida: S01, S02

In [0]:
paso_query95 =  f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_15 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento     
,26                     AS criterio_salida
,CASE 
  WHEN IFNULL(A.IND_S01,0) = 1 AND IFNULL(A.IND_S02,0) = 1 AND IFNULL(A.IND_S14,0) = 0 AND IFNULL(A.IND_N01,0) = 0 AND IFNULL(A.IND_N02,0) = 0 AND IFNULL(A.IND_N03,0) = 0 AND IFNULL(A.IND_N04,0) = 0
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A
    
"""

In [0]:
sql_safe(paso_query95)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_RES_D00_OPE_CRIT_EVAL_15 as
SELECT
 A.periodo_cierre         AS periodo_cierre            
,A.fecha_cierre           AS fecha_cierre          
,A.tipo_proceso           AS tipo_proceso          
,A.rut_cliente            AS rut_cliente         
,A.dv_rut_cliente         AS dv_rut_cliente            
,A.tipo_operacion         AS tipo_operacion            
,A.operacion              AS operacion       
,A.sistema                AS sistema     
,A.segmento               AS segmento     
,26                     AS criterio_salida
,CASE 
  WHEN IFNULL(A.IND_S01,0) = 1 AND IFNULL(A.IND_S02,0) = 1 AND IFNULL(A.IND_S14,0) = 0 AND IFNULL(A.IND_N01,0) = 0 AND IFNULL(A.IND_N02,0) = 0 AND IFNULL(A.IND_N03,0) = 0 AND IFNULL(A.IND_N04,0) = 0
  THEN 1
  ELSE 0
 END                    AS flag_resultado_regla
FROM 
    tmp_RES_matriz_cartdet_crit_sal_ope_eval  A
    



DataFrame[]

### Salida Temporal a Nivel de Campo Evaludado (tmp_tbl_cartdet_crit_ent_crit)
------------------
* generar salida temporal a nivel de campo evaluado. 
* se registran todas las operaciones evaluadas


In [0]:

paso_query250 = f"""
CREATE OR REPLACE TEMPORARY VIEW tmp_tbl_cartdet_crit_sal_crit AS
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_1  
UNION 
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_2  
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_3  
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_4 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_5 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_7 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_8 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_9 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_10 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_11
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_12 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_13
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_14
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_15
"""  

In [0]:
sql_safe(paso_query250)

sql_safe: query -> 
CREATE OR REPLACE TEMPORARY VIEW tmp_tbl_cartdet_crit_sal_crit AS
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_1  
UNION 
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_2  
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_3  
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_4 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_5 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_7 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_8 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_9 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_10 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_11
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_12 
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_13
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_14
UNION
SELECT * FROM   tmp_RES_D00_OPE_CRIT_EVAL_15



DataFrame[]

## Carga Tablas de Salidas
--------------------------------------
* carga resultados a tablas de salidas del notebook

### Carga Tabla Evaluacion 


#### Reproceso (Elimina registros en caso de reprocesos)

In [0]:
paso_query300 = f"""DELETE FROM {base_silver_x}.tbl_cd_cartdet_crit_sal_crit where criterio_salida in {p_critdet} """

In [0]:
sql_safe(paso_query300)

sql_safe: query -> DELETE FROM dsr_gld_bciwork_db.tbl_cd_cartdet_crit_sal_crit where criterio_salida in (2, 27, 62, 40, 41, 67, 60, 61, 63, 64, 65, 66, 30, 26) 


DataFrame[num_affected_rows: bigint]

#### Inserta Registros tabla salida

In [0]:
paso_query310 = f"""
INSERT INTO {base_silver_x}.tbl_cd_cartdet_crit_sal_crit
SELECT 
    IFNULL(periodo_cierre,19000101),
    IFNULL(fecha_cierre,190001),
    IFNULL(tipo_proceso,' '),
    IFNULL(rut_cliente,0),
    IFNULL(dv_rut_cliente,' '),
    IFNULL(tipo_operacion,' '),
    IFNULL(operacion,' '),
    IFNULL(sistema,' '),
    IFNULL(segmento,' '),
    IFNULL(criterio_salida,0),
    IFNULL(flag_resultado_regla,0)
FROM
    tmp_tbl_cartdet_crit_sal_crit  
"""  


In [0]:
sql_safe(paso_query310)

sql_safe: query -> 
INSERT INTO dsr_gld_bciwork_db.tbl_cd_cartdet_crit_sal_crit
SELECT 
    IFNULL(periodo_cierre,19000101),
    IFNULL(fecha_cierre,190001),
    IFNULL(tipo_proceso,' '),
    IFNULL(rut_cliente,0),
    IFNULL(dv_rut_cliente,' '),
    IFNULL(tipo_operacion,' '),
    IFNULL(operacion,' '),
    IFNULL(sistema,' '),
    IFNULL(segmento,' '),
    IFNULL(criterio_salida,0),
    IFNULL(flag_resultado_regla,0)
FROM
    tmp_tbl_cartdet_crit_sal_crit  



DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

## Estadisticas

In [0]:
%sql
select 
fecha_cierre, 
criterio_salida, 
count(1) 
from ${bci.dbnamesilver}.tbl_cd_cartdet_crit_sal_crit  
group by 1,2 order by 1,2

fecha_cierre,criterio_salida,count(1)
20250930,2,324508
20250930,26,324508
20250930,27,324508
20250930,30,324508
20250930,40,324508
20250930,41,324508
20250930,60,324508
20250930,61,324508
20250930,62,324508
20250930,63,324508


## Mensaje termino OK

In [0]:
msgerrorx="OK"
dbutils.notebook.exit("{\"coderror\":0, \"msgerror\":\""+msgerrorx+"\"}")